# Project 4: Automated Blog Post Researcher & Writer (Sequential Multi-Agent Pipeline)

This notebook implements an enterprise-grade Multi-Agent Blog Generation Pipeline featuring:
- **LangGraph StateGraph**: Deterministic state passing across node boundaries using a typed `BlogState` schema.
- **Specialized Multi-Agent Nodes**:
  1. **Researcher Agent**: Gathers real-time web facts, stats, and source URLs via `TavilySearch`.
  2. **Writer Agent**: Synthesizes structured Markdown blog posts with inline citations.
  3. **Fact-Checker Agent**: Audits draft claims using Pydantic structured output (`FactCheckReport`).
  4. **Editor Agent**: Refines and polishes flagged claims based on fact-checker feedback.
- **Conditional Control Loop**: Automated revision loop with a `max_revisions` safety cap to prevent infinite execution.


In [ ]:
import os
from typing import List, Dict, Any, Optional, TypedDict
from pydantic import BaseModel, Field
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, START, END

load_dotenv()

# Verify API Keys
if not os.getenv("GROQ_API_KEY"):
    print("⚠️ GROQ_API_KEY missing from environment.")
if not os.getenv("TAVILY_API_KEY"):
    print("⚠️ TAVILY_API_KEY missing from environment.")

# Initialize Primary LLM (Groq Llama 3.3 70B)
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2)



In [ ]:
# 1. Define Multi-Agent Shared State Schema
class BlogState(TypedDict):
    topic: str                     # Target blog topic
    research_notes: str            # Fact-checked bulleted notes from Tavily
    sources: List[str]             # List of reference URLs
    draft: str                     # Initial draft generated by Writer Agent
    fact_check_passed: bool        # Validation status from Fact-Checker Agent
    fact_check_feedback: str       # Detailed critique / unverified claims report
    final_post: str                # Final published markdown blog post
    revision_count: int            # Active revision count (max 2)

print("BlogState schema defined successfully!")



In [ ]:
# 2. Researcher Agent Node
tavily_tool = TavilySearch(max_results=3)

def research_node(state: BlogState) -> Dict[str, Any]:
    topic = state["topic"]
    print(f"🔎 [RESEARCHER AGENT] Searching web for topic: '{topic}'...")
    
    # Run Tavily Search
    search_results = tavily_tool.invoke({"query": topic})
    
    # Extract raw content and sources
    raw_texts = []
    sources = []
    
    if isinstance(search_results, dict) and "results" in search_results:
        items = search_results["results"]
    elif isinstance(search_results, list):
        items = search_results
    else:
        items = []
        
    for item in items:
        if isinstance(item, dict):
            content = item.get("content", "")
            url = item.get("url", "")
            if content:
                raw_texts.append(content)
            if url:
                sources.append(url)
                
    combined_raw = "\n\n".join(raw_texts) if raw_texts else "No search results returned."
    
    # Synthesize structured research notes via LLM
    prompt = f"""You are an expert Lead Technical Researcher.
Analyze the following raw web search results for the topic '{topic}' and extract key factual insights.

RAW SEARCH DATA:
{combined_raw}

OUTPUT REQUIREMENTS:
- Provide 5-8 clear, factual, bulleted research points.
- Include specific data points, statistics, or technological details mentioned in the search data.
- Do NOT invent facts not present in the search data.
"""
    response = llm.invoke([{"role": "user", "content": prompt}])
    research_notes = response.content
    
    print(f"✅ [RESEARCHER AGENT] Extracted {len(sources)} source references.")
    return {
        "research_notes": research_notes,
        "sources": sources,
        "revision_count": 0
    }



In [ ]:
# 3. Writer Agent Node
def writer_node(state: BlogState) -> Dict[str, Any]:
    topic = state["topic"]
    research_notes = state["research_notes"]
    sources = state["sources"]
    
    print(f"✍️ [WRITER AGENT] Drafting blog post for '{topic}'...")
    
    formatted_sources = "\n".join([f"- {url}" for url in sources])
    
    prompt = f"""You are a Senior Tech Journalist and Blog Author.
Write an engaging, high-quality Markdown blog post based strictly on the provided research notes.

TOPIC: {topic}

RESEARCH NOTES:
{research_notes}

AVAILABLE SOURCES:
{formatted_sources}

BLOG STRUCTURE REQUIREMENTS:
1. **Catchy Title** (H1 header).
2. **Engaging Introduction**: Hook the reader and define the core problem/innovation.
3. **Core Sections** (H2/H3 headers): Detailed, well-written analysis incorporating the research notes.
4. **Conclusion & Key Takeaways**.
5. **References Section**: List the provided source URLs cleanly.

STYLE REQUIREMENTS:
- Maintain an authoritative yet conversational tech journalism tone.
- Explicitly cite source URLs in text where relevant (e.g. [Source](url)).
- Do NOT make claims outside the research notes.
"""
    response = llm.invoke([{"role": "user", "content": prompt}])
    draft = response.content
    
    print("✅ [WRITER AGENT] Initial draft completed!")
    return {"draft": draft}



In [ ]:
# 4. Fact-Checker Agent Node
class FactCheckReport(BaseModel):
    is_accurate: bool = Field(description="True if the draft accurately reflects research notes without ungrounded claims, False otherwise.")
    unverified_claims: List[str] = Field(default=[], description="List of specific statements in draft not supported by research notes.")
    suggestions: str = Field(description="Actionable editing instructions for the Editor Agent.")

fact_checker_llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.0).with_structured_output(FactCheckReport)

def fact_checker_node(state: BlogState) -> Dict[str, Any]:
    draft = state["draft"]
    research_notes = state["research_notes"]
    
    print("🛡️ [FACT-CHECKER AGENT] Auditing draft accuracy against research notes...")
    
    prompt = f"""You are a Lead Fact-Checker and Editorial Quality Auditor.
Audit the following blog post draft against the official research notes.

RESEARCH NOTES:
{research_notes}

BLOG POST DRAFT:
{draft}

AUDIT RULES:
- Verify that statistics, quotes, and technical assertions in the draft match the research notes.
- Flag any ungrounded claims or hallucinated facts as unverified_claims.
- Set is_accurate = True if all core claims are grounded in research notes.
- Set is_accurate = False if there are notable ungrounded claims requiring correction.
"""
    report: FactCheckReport = fact_checker_llm.invoke([{"role": "user", "content": prompt}])
    
    if report.is_accurate:
        print("✅ [FACT-CHECKER AGENT] Draft PASSED fact-checking audit!")
        feedback = "Fact-check passed cleanly. No major unverified claims found."
    else:
        print(f"⚠️ [FACT-CHECKER AGENT] Draft FLAGGED with {len(report.unverified_claims)} unverified claims.")
        feedback = f"Unverified claims: {report.unverified_claims}\nSuggestions: {report.suggestions}"
        
    return {
        "fact_check_passed": report.is_accurate,
        "fact_check_feedback": feedback,
        "final_post": draft  # Default to current draft if passed
    }



In [ ]:
# 5. Editor Agent Node
def editor_node(state: BlogState) -> Dict[str, Any]:
    draft = state["draft"]
    feedback = state["fact_check_feedback"]
    count = state.get("revision_count", 0) + 1
    
    print(f"🎨 [EDITOR AGENT] Polishing & revising draft (Iteration #{count})...")
    
    prompt = f"""You are an Executive Editor polishing a blog post draft based on Fact-Checker feedback.

CURRENT DRAFT:
{draft}

FACT-CHECKER FEEDBACK:
{feedback}

REVISION REQUIREMENTS:
- Correct or remove any unverified claims flagged by the fact-checker.
- Refine formatting, header flow, and readability.
- Maintain all valid citations and references.
- Return the complete, polished Markdown blog post.
"""
    response = llm.invoke([{"role": "user", "content": prompt}])
    polished_post = response.content
    
    print("✅ [EDITOR AGENT] Revision completed!")
    return {
        "final_post": polished_post,
        "revision_count": count
    }



In [ ]:
# 6. Build Multi-Agent StateGraph Workflow
def should_revise(state: BlogState) -> str:
    # Conditional Edge: Determines whether to route to Editor Agent or END
    passed = state.get("fact_check_passed", True)
    revisions = state.get("revision_count", 0)
    
    if not passed and revisions < 2:
        print(f"🔄 Routing to EDITOR AGENT (Revision {revisions + 1}/2)...")
        return "editor"
    else:
        print("🏁 Workflow complete! Routing to END.")
        return END

# Build State Graph
builder = StateGraph(BlogState)

# Add Nodes
builder.add_node("researcher", research_node)
builder.add_node("writer", writer_node)
builder.add_node("fact_checker", fact_checker_node)
builder.add_node("editor", editor_node)

# Add Edges
builder.add_edge(START, "researcher")
builder.add_edge("researcher", "writer")
builder.add_edge("writer", "fact_checker")

# Add Conditional Edge from Fact-Checker
builder.add_conditional_edges("fact_checker", should_revise, {
    "editor": "editor",
    END: END
})

builder.add_edge("editor", END)

# Compile Graph
graph = builder.compile()
print("Multi-Agent StateGraph compiled successfully!")
graph



In [ ]:
from IPython.display import display, Markdown
# 7. Execute Multi-Agent Pipeline Test Run
topic = "Impact of Small Language Models (SLMs) on Edge AI in 2026"

print(f"🚀 Starting Automated Multi-Agent Pipeline for Topic: '{topic}'\n" + "="*70)

initial_state = {"topic": topic, "revision_count": 0}
final_state = graph.invoke(initial_state)

print("\n" + "="*70 + "\n🎉 PIPELINE COMPLETE! FINAL PUBLISHED BLOG POST:\n" + "="*70 + "\n")
display(Markdown(final_state["final_post"]))



In [ ]:
topic = "How is loop engineering being incorporated into agentic systems in production?"

print(f"🚀 Starting Automated Multi-Agent Pipeline for Topic: '{topic}'\n" + "="*70)

initial_state = {"topic": topic, "revision_count": 0}
final_state = graph.invoke(initial_state)

print("\n" + "="*70 + "\n🎉 PIPELINE COMPLETE! FINAL PUBLISHED BLOG POST:\n" + "="*70 + "\n")
display(Markdown(final_state["final_post"]))